In [1]:
import json, os, re
import pandas as pd
import sqlite3
import shutil
import numpy as np
from datetime import datetime

# Parsing Google Keep Notes into Workout JSON files

In [2]:
# Initialize data path
path = 'Data'
data_path = 'Workout logs'

## Filtering the workout logs notes

In [13]:
# Make the workout logs folder
if not os.path.exists(data_path):
    os.makedirs(data_path)

# Iterate all the files if there's no content in workout logs folder
if len(os.listdir(data_path)) == 0:
    for filepath in os.listdir(path):
        # Read the file
        full_filepath = os.path.join(path, filepath)
        file = open(full_filepath).read()
        # Load file as json
        json_file = json.loads(file)

        # Check if file is a Workout Log
        label = 'Workout Log'
        if 'labels' in json_file and {'name': label} in json_file['labels']:
            shutil.copy(full_filepath, data_path)
            print(f'Copied {filepath} to {data_path}')

Copied Apr 18(2).json to Workout logs
Copied Apr 2, 2025.json to Workout logs
Copied Apr 22.json to Workout logs
Copied Apr 24, 2025.json to Workout logs
Copied Apr 24.json to Workout logs
Copied Apr 26, 2024.json to Workout logs
Copied Apr 28.json to Workout logs
Copied Apr 29, 2024.json to Workout logs
Copied Apr 4.json to Workout logs
Copied Apr 6, 2025.json to Workout logs
Copied Apr 6.json to Workout logs
Copied Apr 7, 2025.json to Workout logs
Copied Apr 8.json to Workout logs
Copied Apr 9, 2024.json to Workout logs
Copied Apr. 24, 2024.json to Workout logs
Copied April 12, 2024.json to Workout logs
Copied April 16, 2024.json to Workout logs
Copied April 19, 2024.json to Workout logs
Copied April 21, 2024.json to Workout logs
Copied April 3, 2023.json to Workout logs
Copied April 3, 2025.json to Workout logs
Copied April 4, 2024.json to Workout logs
Copied Aug 11.json to Workout logs
Copied Aug 13.json to Workout logs
Copied Aug 14.json to Workout logs
Copied Aug 17, 2024.json to

## Renaming workout logs notes

In [35]:
# Iterate all the files and check the title if it has a year
for filepath in os.listdir(data_path):
    full_filepath = os.path.join(data_path, filepath)
    with open(full_filepath) as f:
        json_file = json.load(f)

    # Get the title
    if 'title' in json_file:
        try:
            # Get year
            pd.Timestamp(json_file['title']).year
            print(f'Skipped renaming {filepath}')
        except:
            # Get the creation timestamp
            if 'createdTimestampUsec' in json_file:
                timestamp = json_file['createdTimestampUsec']
                # Convert the timestamp to a BB dd, YYYY
                new_title = datetime.fromtimestamp(int(timestamp) / 1_000_000).strftime('%B %d, %Y')

                # Auto-increment suffix if file already exists
                new_filename = f'{new_title}.json'
                new_filepath = os.path.join(data_path, new_filename)
                counter = 1
                while os.path.exists(new_filepath):
                    new_filename = f'{new_title}({counter}).json'
                    new_filepath = os.path.join(data_path, new_filename)
                    counter += 1

                # Rename the path
                os.rename(full_filepath, new_filepath)
                print(f'Renamed: {filepath} -> {new_filename}')

Skipped renaming Apr 2, 2025.json
Skipped renaming Apr 24, 2025.json
Renamed: Apr 24.json -> April 24, 2022(1).json
Skipped renaming Apr 26, 2024.json
Renamed: Apr 28.json -> April 28, 2022.json
Skipped renaming Apr 29, 2024.json
Renamed: Apr 4.json -> April 04, 2022.json
Skipped renaming Apr 6, 2025.json
Renamed: Apr 6.json -> April 06, 2022.json
Skipped renaming Apr 7, 2025.json
Renamed: Apr 8.json -> April 08, 2022.json
Skipped renaming Apr 9, 2024.json
Skipped renaming Apr. 24, 2024.json
Skipped renaming April 12, 2024.json
Skipped renaming April 16, 2024.json
Renamed: April 18, 2022(2).json -> April 18, 2022.json
Skipped renaming April 19, 2024.json
Skipped renaming April 21, 2024.json
Renamed: April 24, 2022.json -> April 24, 2022(2).json
Skipped renaming April 3, 2023.json
Skipped renaming April 3, 2025.json
Skipped renaming April 4, 2024.json
Renamed: Aug 11.json -> August 11, 2022.json
Renamed: Aug 13.json -> August 13, 2022.json
Renamed: Aug 14.json -> August 14, 2022.json
Sk

In [28]:
pd.Timestamp('Feb 13, 2020').year

2020

In [33]:
datetime.fromtimestamp(1739880806652000/1_000_000).strftime('%B %d, %Y')

'February 18, 2025'

## Making the parser

### Trial 1

In [4]:
# Read the file 
with open(os.path.join(path, 'Feb 18, 2025.json')) as f:
    file = f.read()

# Load as json
json_file = json.loads(file)
json_file

{'color': 'DEFAULT',
 'isTrashed': False,
 'isPinned': False,
 'isArchived': False,
 'textContent': 'Bench press - 12.5 lbs 10, 15 lbs 10, 17.5 lbs 7 no rest 15 4, 20 lbs 3 no rest 17.5 1 no rest 15 lbs 2\nCable fly (lower) - 22 lbs 12 , 33 lbs 10, 33 lbs 10\nInclined dumbbell fly - 15 lbs 10, 10, 20 lbs 10\nPec deck fly (lower) - 60 lbs 10, 70 lbs>60 lbs 7>6, 6>7\nArnold machine - 5 lbs 8, 6, \nTricep dip machine - 2x35 lbs 10, 2x(35+2.5 lbs) 10, 2x35 lbs + 4x2.5 lbs 10\nChest machine - 30 lbs 10, 45 lbs>30 lbs 2 then 8>10, 7>5\nTricep pulldown - 55 lbs>44 lbs 7F>5F, 7F>4F, 7F>3F\n\n\nWeight - 76 kg\n\n',
 'title': 'Feb 18, 2025',
 'userEditedTimestampUsec': 1739880806652000,
 'createdTimestampUsec': 1739875379471000,
 'textContentHtml': '<p dir="ltr" style="line-height:1.38;margin-top:0.0pt;margin-bottom:0.0pt;"><span style="font-size:7.2pt;font-family:\'Google Sans\';color:#000000;background-color:transparent;font-weight:400;font-style:normal;font-variant:normal;text-decoration:none

In [5]:
json_file['textContent']

'Bench press - 12.5 lbs 10, 15 lbs 10, 17.5 lbs 7 no rest 15 4, 20 lbs 3 no rest 17.5 1 no rest 15 lbs 2\nCable fly (lower) - 22 lbs 12 , 33 lbs 10, 33 lbs 10\nInclined dumbbell fly - 15 lbs 10, 10, 20 lbs 10\nPec deck fly (lower) - 60 lbs 10, 70 lbs>60 lbs 7>6, 6>7\nArnold machine - 5 lbs 8, 6, \nTricep dip machine - 2x35 lbs 10, 2x(35+2.5 lbs) 10, 2x35 lbs + 4x2.5 lbs 10\nChest machine - 30 lbs 10, 45 lbs>30 lbs 2 then 8>10, 7>5\nTricep pulldown - 55 lbs>44 lbs 7F>5F, 7F>4F, 7F>3F\n\n\nWeight - 76 kg\n\n'

In [3]:
movements = '12.5 lbs 10, 15 lbs 10, 17.5 lbs 7 no rest 15 4, 20 lbs 3 no rest 17.5 1 no rest 15 lbs 2'

In [6]:
def parse_movement(text):
    # Split text by comma and any number of whitespaces
    sets = re.split(r',\s*', text)

    return sets

In [7]:
parse_movement(movements)

['12.5 lbs 10',
 '15 lbs 10',
 '17.5 lbs 7 no rest 15 4',
 '20 lbs 3 no rest 17.5 1 no rest 15 lbs 2']

In [8]:
def find_rightmost_weight(text):
    # Initialize index and units of weight
    weight_index = -1
    weight_units = ['kg', 'lbs', 'bar', 'caret', 'BW']
    rightmost_weight = None

    for weight_unit in weight_units:
            print(f'Looking for {weight_unit}')
            # Find the weight unit
            found_index = text.rfind(weight_unit)

            # Update weight index if it's the rightmost weight unit
            if found_index > weight_index:
                print(f'Found {weight_unit} at {found_index}')
                weight_index = found_index
                rightmost_weight = weight_unit

    print(f'Rightmost weight unit is {rightmost_weight} at {weight_index}')

    return weight_index, rightmost_weight

In [9]:
def separate_load_from_rep(text):
    # Get index of rightmost weight
    weight_index, rightmost_weight = find_rightmost_weight(text)

    # Early return if no weight index
    if weight_index == -1:
        return None, text

    # Initialize boundary accounting for length of unit
    boundary_index = weight_index + len(rightmost_weight)
    # Check for the actual boundary between load and rep
    for t in text[boundary_index:]:
        try: 
            # Convert character to integer
            int(t)
            # Increment boundary index
            boundary_index += 1
        except:
            break

    # Separate load from rep
    load = text[:boundary_index].strip()
    rep = text[boundary_index:].strip()
    
    return load, rep

In [10]:
separate_load_from_rep('10')

Looking for kg
Looking for lbs
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is None at -1


(None, '10')

In [11]:
DROPSET_IDS = ['no rest', '...']

In [13]:
def is_set_unit(text):
    # Initialize that set is a unit set
    is_unit_set = True
    # Initialize identifiers for no-break/drop sets
    dropset_ids = DROPSET_IDS
    # Check if set is unit or no-break/drop
    if any([dropset_id in text for dropset_id in dropset_ids]):
        is_unit_set = False

    return is_unit_set

In [14]:
def parse_subset_hand(text):
    # Split left and right rep
    left_rep, right_rep = text.split('~')

    return {
        'left': left_rep,
        'right': right_rep
    }

In [20]:
def parse_subset(text, is_unit_set=True):
    if is_unit_set:
        if isinstance(text, int):
            return text
        # Check for left and right-specific reps
        if '~' in text:
            return parse_subset_hand(text)
        
        # Convert text to integer to check if there is no rep remark
        try:
            return int(text)
        except:
            return text
    else:
        # Initialize identifiers for dropset
        dropset_ids = DROPSET_IDS

        # Initialize separated dropsets
        result = []
        # Iterate over all subsets
        for dropset_id in dropset_ids:
            if dropset_id in text:
                # Split text by dropset identifier
                result = text.split(dropset_id)
                # Strip all text
                result = [subset.strip() for subset in result]

            # Parse each subset individually if there are left and right-specific rep
            result = [parse_subset(subset) for subset in result]

        return result

In [16]:
def parse_set(text, previous_load=None):
    # Separate load from rep
    load, rep = separate_load_from_rep(text)
    # Check if rep is unit or drop
    is_unit_set = is_set_unit(rep)
    print(f'{text} is a {"unit" if is_unit_set else "drop"} set!')

    if previous_load:
        load = previous_load

    # Initialize storage
    result = {}

    # Add loads and reps to result
    result['kind'] = 'unit' if is_unit_set else 'drop'
    result['subsets'] = {}

    # Add load to subset if drop set or not
    result['subsets']['load'] = load
    result['subsets']['reps'] = parse_subset(rep, is_unit_set)

    return result

In [17]:
parse_set('12.5 lbs 10...5...2')

Looking for kg
Looking for lbs
Found lbs at 5
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 5
12.5 lbs 10...5...2 is a drop set!


{'kind': 'drop', 'subsets': {'load': '12.5 lbs', 'reps': [10, 5, 2]}}

In [58]:
# text = movements

# # Initialize index and units of weight
# weight_index = 0
# weight_units = ['kg', 'lbs', 'bar', 'caret', 'BW']
# # Initialize that unit can be found
# weight_unit_found = [True, True, True, True, True]

# while any(weight_unit_found):
#     for idx, weight_unit in enumerate(weight_units):
#         # Stop looking for weight units that cannot be found
#         if not weight_unit_found[idx]:
#             continue

#         print(f'Looking for {weight_unit}')
#         # Find the weight unit
#         found_index = text.find(weight_unit)

#         # Update weight index if it's the rightmost weight unit
#         if found_index > weight_index:
#             print(f'Found {weight_unit} at {found_index}')
#             weight_index = found_index

#         if found_index == -1:
#             print(f'{weight_unit} cannot be found')
#             weight_unit_found[idx] = False

In [59]:
# # Initialize index and units of weight
# weight_index = 0
# weight_units = ['kg', 'lbs', 'bar', 'caret', 'BW']

# for idx, weight_unit in enumerate(weight_units):
#         print(f'Looking for {weight_unit}')
#         # Find the weight unit
#         found_index = text.rfind(weight_unit)

#         # Update weight index if it's the rightmost weight unit
#         if found_index > weight_index:
#             print(f'Found {weight_unit} at {found_index}')
#             weight_index = found_index

# print(f'Found rightmost weight unit at {weight_index}')

In [18]:
def parse_exercise(text):
    # Initialize storage
    result_dict = {}

    # Get the name by splitting by hyphen
    name, movements = re.split(r'-\s*', text)
    # Clean name by whitespace
    name = name.strip()

    # Make id
    id = name.lower().replace(' ', '_')
    result_dict['id'] = id

    # Add the movements to result dictionary
    result_dict['movements'] = []

    # Add movement name and sets
    movement_dict = {'name': name, 'sets': []}
    result_dict['movements'].append(movement_dict)

    # Get all sets
    sets = re.split(r',\s*', movements)
    for set_text in sets:
        result_dict['movements'][0]['sets'].append(parse_set(set_text))

    return result_dict

In [21]:
parse_exercise('Bench press - 12.5 lbs 10, 15 lbs 10, 17.5 lbs 7 no rest 15 4, 20 lbs 3 no rest 17.5 1 no rest 15 lbs 2')

Looking for kg
Looking for lbs
Found lbs at 5
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 5
12.5 lbs 10 is a unit set!
Looking for kg
Looking for lbs
Found lbs at 3
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 3
15 lbs 10 is a unit set!
Looking for kg
Looking for lbs
Found lbs at 5
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 5
17.5 lbs 7 no rest 15 4 is a drop set!
Looking for kg
Looking for lbs
Found lbs at 35
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 35
20 lbs 3 no rest 17.5 1 no rest 15 lbs 2 is a unit set!


{'id': 'bench_press',
 'movements': [{'name': 'Bench press',
   'sets': [{'kind': 'unit', 'subsets': {'load': '12.5 lbs', 'reps': 10}},
    {'kind': 'unit', 'subsets': {'load': '15 lbs', 'reps': 10}},
    {'kind': 'drop', 'subsets': {'load': '17.5 lbs', 'reps': [7, '15 4']}},
    {'kind': 'unit',
     'subsets': {'load': '20 lbs 3 no rest 17.5 1 no rest 15 lbs',
      'reps': 2}}]}]}

In [19]:
def parse_text_content(text):
    # Split string by new lines
    result = text.split('\n')

    return result

In [20]:
parse_text_content(json_file['textContent'])

['Bench press - 12.5 lbs 10, 15 lbs 10, 17.5 lbs 7 no rest 15 4, 20 lbs 3 no rest 17.5 1 no rest 15 lbs 2',
 'Cable fly (lower) - 22 lbs 12 , 33 lbs 10, 33 lbs 10',
 'Inclined dumbbell fly - 15 lbs 10, 10, 20 lbs 10',
 'Pec deck fly (lower) - 60 lbs 10, 70 lbs>60 lbs 7>6, 6>7',
 'Arnold machine - 5 lbs 8, 6, ',
 'Tricep dip machine - 2x35 lbs 10, 2x(35+2.5 lbs) 10, 2x35 lbs + 4x2.5 lbs 10',
 'Chest machine - 30 lbs 10, 45 lbs>30 lbs 2 then 8>10, 7>5',
 'Tricep pulldown - 55 lbs>44 lbs 7F>5F, 7F>4F, 7F>3F',
 '',
 '',
 'Weight - 76 kg',
 '',
 '']

In [ ]:
# Iterate over all files
for filename in os.listdir(path):
    # Open file
    string_file = open(os.path.join(path, filename)).read()
    # Parse to json
    